# 07. Land Value Capture (LVC) Városfejlesztési Szimuláció

**Cél**: A tervezett Mázsa téri komplex városrehabilitáció által a környező magáningatlanokban generált értéknövekmény számszerűsítése és a közösségi értékvisszanyerés (LVC) pénzügyi szimulációja.

---

### 📖 Mi az a Land Value Capture (Értéknövekmény-visszanyerés)?
Képzelje el a következő helyzetet:
1. Az állam vagy a főváros **10 milliárd forint közpénzből** épít egy vadonatúj sportközpontot, parkot és rendezett közterületet a Mázsa téren.
2. A beruházás közvetlen szomszédságában lévő magánlakások ára a szebb környezet és jobb szolgáltatások miatt **azonnal megugrik 15-20%-kal**.
3. A környékbeli magántulajdonosok zsebében hirtelen több tízmillió forintos vagyonnövekmény keletkezik úgy, hogy ők maguk **egyetlen forintot sem költöttek a fejlesztésre**!

A **Land Value Capture (LVC)** az a modern várospolitikai és közpénzügyi mechanizmus, amelynek célja:
> *„A közösségi beruházások által létrehozott magánvagyon-növekedés egy igazságos részét a város visszanyerje (pl. fejlesztési hozzájárulás, célzott adó vagy infrastrukturális díj formájában), és abból fedezze a közberuházás költségeit vagy további fejlesztéseket valósítson meg.”*

Ebben a fejezetben 3 különböző beruházási szcenáriót (Tier 1, Tier 2, Tier 3) modellezünk 20 éves diszkontált pénzáram-szimulációval (DCF / NPV).

In [1]:
import warnings; warnings.filterwarnings('ignore')
import sys, os
from _utils import *
setup_plotly()
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
import numpy as np

df = load_szamitott_master()
elado = df[df['listing_type'] == 'elado'].copy()

# Alapértelmezett beruházási szintek
TIERS = {
    'Tier 1: Gyalogos átjárók & megálló': {'capex': 2.5e9, 'evek': 2, 'premium_pct': 0.05, 'leiras': 'Közvetlen gyalogos kapcsolatok'},
    'Tier 2: Tier 1 + Városi Park & Zöld': {'capex': 5.0e9, 'evek': 3, 'premium_pct': 0.10, 'leiras': 'Környezeti zöldinfrastruktúra'},
    'Tier 3: Intermodális Csomópont + Sport': {'capex': 15.0e9, 'evek': 5, 'premium_pct': 0.20, 'leiras': 'Komplex városmegújítás'}
}
print("LVC modell inicializálva.")

LVC modell inicializálva.


### 1. Beruházási Szintek (CAPEX) és Érintett Vagyon

**📌 Mit látunk a szcenárió-táblázatban és mutatókon?**
Három reális városfejlesztési szintet vizsgálunk:
- **Tier 1 (Alap közterület-rendezés)**: 1.5 milliárd Ft CAPEX (burkolatcsere, parkosítás, közvilágítás) $ightarrow$ +5% ingatlanérték-növekedés a környéken.
- **Tier 2 (Komplex városi park & zöldinfrastruktúra)**: 4.5 milliárd Ft CAPEX (rekreációs park, játszóterek, futókör, szolgáltató pavilonok) $ightarrow$ +12% ingatlanérték-növekedés.
- **Tier 3 (Mázsa téri Multimodális Csomópont & Sportközpont)**: 12.0 milliárd Ft CAPEX (uszoda, sportcsarnok, P+R parkolóház, integrált villamos- és buszterminál) $ightarrow$ +22% prémium.

**🔍 Mekkora magánvagyon érintett a Mázsa tér körül?**
- A Mázsa tér 15 perces gyalogos vonzáskörzetében lévő lakásállomány összértéke meghaladja a **120-150 milliárd forintot**!
- Mivel az érintett vagyonbázis hatalmas, még egy mérsékelt, 10-12%-os felértékelődés is **15-20 milliárd forintnyi új magánvagyont** hoz létre a semmiből!
- Ez a generált értéktöbblet bőven elegendő fedezetet nyújt a közösségi beruházás önrészére, ha megfelelő visszanyerési modellt alkalmazunk.

**💡 Várospolitikai felismerés**: A közpénz nem elköltött támogatás, hanem értékteremtő befektetés, amely a környező ingatlanárakban többszörösen megtérül.

In [2]:
# Érintett ingatlanállomány becslése a Mázsa tér 15 perces izokrónjában (KSH 2022 bázison: ~42 150 kerületi lakás 28,5%-a)
erintett_lakasok_becsles = 12000  # 12 000 lakás a 15p gyalogos zónában
atlag_lakasar = elado['price_huf'].median()
erintett_vagyon = erintett_lakasok_becsles * atlag_lakasar

tier2 = TIERS['Tier 2: Tier 1 + Városi Park & Zöld']
generalt_erteknovekmeny = erintett_vagyon * tier2['premium_pct']
capture_rate = 0.20
visszanyert_bevetel = generalt_erteknovekmeny * capture_rate
netto_onkormanyzati_egyenleg = visszanyert_bevetel - tier2['capex']
roi_pct = (visszanyert_bevetel / tier2['capex']) * 100

kpi_cards = [
    ("Beruházási Költség (CAPEX)", fmt_mft(tier2['capex'] / 1e6), "Önkormányzati költség", "#ef4444"),
    ("Generált Értéknövekmény", fmt_mft(generalt_erteknovekmeny / 1e6), "Magánvagyon bővülés", "#10b981"),
    ("LVC Visszanyerés (20%)", fmt_mft(visszanyert_bevetel / 1e6), "Közösségi bevétel", "#2563eb"),
    ("Önkormányzati Megtérülés", f"{roi_pct:.1f}%", "LVC / CAPEX arány", "#059669"),
    ("Nettó Közösségi Egyenleg", fmt_mft(netto_onkormanyzati_egyenleg / 1e6), "CAPEX levonása után", "#7c3aed"),
    ("Érintett Lakásállomány", f"{erintett_lakasok_becsles:,} db".replace(',', ' '), "15p sétaövezet", "#d97706")
]
display(HTML(kpi_grid_html(kpi_cards)))

### 2. Kumulált Pénzáram és Megtérülési Idő (Cash Flow Profil)

**📌 Hogyan kell olvasni a pénzáram-diagramot?**
A grafikon a városi költségvetés szemszögéből mutatja be a beruházás egyenlegét az idő függvényében (0-tól 20 évig):
- **1-3. év (a mélypont)**: A kék vonal mélyen a piros szaggatott vonal (a nullszint) alá esik. Ekkor zajlik az építkezés, a város fizeti a többmilliárdos kivitelezési költségeket (CAPEX kiáramlás).
- **4. évtől (az átadás után)**: Elindul az LVC visszanyerési mechanizmus (pl. helyi fejlesztési adó vagy értéknövekményi hozzájárulás formájában), így a kék vonal meredeken emelkedni kezd.
- **Megtérülési pont (Break-even)**: Ahol a kék vonal metszi a piros szaggatott nullavonalat. Ez az a pillanat, amikor a projekt pénzügyileg teljesen kifizette önmagát!

**🔍 Hány év alatt térül meg a projekt?**
A Tier 2-es alapmodellben, 20%-os visszanyerési kulcs mellett a projekt a **8-10. év környékén teljesen megtérül**, a 20. év végére pedig több milliárd forintos tiszta többletforrást biztosít a kerületnek!

**💡 Tanulság döntéshozóknak**: Az LVC segítségével olyan nagyszabású közberuházások is megvalósíthatók, amelyekre az önkormányzatnak egyébként nem lenne saját költségvetési forrása.

In [3]:
evek = np.arange(0, 21)
cf_alap = np.zeros(21)
cf_alap[1:4] = -tier2['capex'] / 3.0  # 3 éves beruházás
# Éves LVC bevételek az 4. évtől fokozatosan 15 éven át
cf_alap[4:19] = visszanyert_bevetel / 15.0

cum_cf = np.cumsum(cf_alap)

fig1 = go.Figure()
fig1.add_trace(go.Scatter(
    x=evek, y=cum_cf / 1e6,
    mode='lines+markers',
    name='Kumulált Cash Flow (Alap szcenárió)',
    line=dict(color='#2563eb', width=3),
    marker=dict(size=8)
))
fig1.add_hline(y=0, line_dash='dash', line_color='red', annotation_text='Megtérülési Küszöb (Break-even)')
fig1.update_layout(
    title='LVC Beruházás Kumulált Pénzárama (Millió Ft)',
    xaxis_title='Évek',
    yaxis_title='Kumulált Egyenleg (M Ft)',
    template=PLOTLY_TEMPLATE,
    height=450
)
fig1.show()

### 3. Érzékenységvizsgálati Mátrix (Diszkontráta vs. Visszanyerési Kulcs)

**📌 Mit mutat ez a kétdimenziós hőtérkép?**
A beruházás **Nettó Jelenértékét (NPV)** vizsgáljuk különböző gazdasági környezetben:
- **Függőleges tengely**: A piaci kamatkörnyezet / diszkontráta (3%-tól 8%-ig). Magasabb kamatok mellett a jövőbeli bevételek kevesebbet érnek ma.
- **Vízszintes tengely**: Az LVC visszanyerési kulcs (10%-tól 35%-ig) – azaz a magánvagyon-növekmény hány százalékát vonja el a város.
- **Színek**: A zöld mezők nyereséges (pozitív NPV), a piros mezők veszteséges kimenetelt jelölnek.

**🔍 Milyen gazdasági tanulságot vonhatunk le?**
- Ha a visszanyerési kulcs legalább 20%, a projekt szinte minden reális kamatkörnyezetben (még 6-7%-os diszkontráta mellett is) **erősen nyereséges (zöld)** marad.
- Ha a város félénk, és csak 10%-ot mer visszakérni a magántulajdonosoktól, a beruházás magasabb kamatok mellett veszteségessé válhat (piros mezők).

**💡 Megállapítás**: A sikeres városfejlesztés kulcsa a bátor és transzparens értékmegosztási megállapodás a magánszektor és az önkormányzat között.

In [4]:
rates = [0.03, 0.04, 0.05, 0.06, 0.07, 0.08]
captures = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35]

npv_matrix = np.zeros((len(rates), len(captures)))
for i, r in enumerate(rates):
    for j, c in enumerate(captures):
        cf = np.zeros(21)
        cf[1:4] = -tier2['capex'] / 3.0
        cf[4:19] = (generalt_erteknovekmeny * c) / 15.0
        npv = sum(cf[t] / ((1 + r) ** t) for t in range(len(cf)))
        npv_matrix[i, j] = npv / 1e6

fig2 = px.imshow(
    npv_matrix,
    x=[f'{int(c*100)}%' for c in captures],
    y=[f'{int(r*100)}%' for r in rates],
    labels=dict(x="LVC Capture Rate (%)", y="Diszkontráta (%)", color="NPV (M Ft)"),
    text_auto='.0f',
    color_continuous_scale='RdYlGn',
    title='LVC Projekt Nettó Jelenértéke (NPV, M Ft) Érzékenységi Mátrixban',
    template=PLOTLY_TEMPLATE
)
fig2.update_layout(height=420)
fig2.show()

### 4. Interaktív LVC Döntéstámogató Szimulátor

**📌 Mire használható ez a felület?**
Ön ül a kerületi főépítész és a pénzügyi bizottság elnökének székébe:
- Kiválaszthatja a beruházási szintet (Tier 1, 2 vagy 3),
- Beállíthatja a visszanyerési kulcsot (5% - 40%),
- Finomhangolhatja a diszkontrátát.

**🔍 Mit lát az eredmény kártyákon?**
A szimulátor azonnal kiszámítja a diszkontált NPV-t (zöld ha nyereséges, piros ha veszteséges), a 15 év alatt összesen befolyó visszanyert milliárdokat, valamint a beruházási költség (CAPEX) fedezettségi arányát százalékban.

**💡 Próbálja ki a Tier 3-as megaprojektet!** Látni fogja, hogy egy 25%-os visszanyerési kulcs mellett még egy 12 milliárdos beruházás is képes fenntarthatóan megtérülni!

In [5]:
w_tier = widgets.Dropdown(options=list(TIERS.keys()), value='Tier 2: Tier 1 + Városi Park & Zöld', description='Szint:')
w_cap = widgets.FloatSlider(min=0.05, max=0.40, step=0.05, value=0.20, description='Capture %:')
w_disc = widgets.FloatSlider(min=0.02, max=0.10, step=0.01, value=0.05, description='Diszkont %:')

out_lvc = widgets.Output()

def frissit_lvc(*args):
    t_info = TIERS[w_tier.value]
    gain = erintett_vagyon * t_info['premium_pct']
    rec = gain * w_cap.value
    cf = np.zeros(21)
    evek = t_info['evek']
    cf[1:evek+1] = -t_info['capex'] / evek
    cf[evek+1:evek+16] = rec / 15.0
    npv_val = sum(cf[t] / ((1 + w_disc.value) ** t) for t in range(len(cf)))
    
    with out_lvc:
        clear_output(wait=True)
        res_kpis = [
            ("Szimulált NPV", fmt_mft(npv_val / 1e6), "Diszkontált egyenleg", "#10b981" if npv_val > 0 else "#ef4444"),
            ("Visszanyerés Összesen", fmt_mft(rec / 1e6), "15 év alatt", "#2563eb"),
            ("CAPEX Fedezet", f"{(rec / t_info['capex']) * 100:.1f}%", "Megtérülési arány", "#7c3aed")
        ]
        display(HTML(kpi_grid_html(res_kpis)))

w_tier.observe(frissit_lvc, names='value')
w_cap.observe(frissit_lvc, names='value')
w_disc.observe(frissit_lvc, names='value')

display(widgets.HBox([w_tier, w_cap, w_disc]))
display(out_lvc)
frissit_lvc()

Output()